<h1> Libraries </h1>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

import outlines #https://dottxt-ai.github.io/outlines/latest/
from pydantic import BaseModel
from typing import List

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print (device)

<h1> Load Model </h1>

In [ ]:
MODEL_NAME = "x"
MODEL_PATH = "y"

modelo = AutoModelForCausalLM.from_pretrained (MODEL_NAME, device_map = device)
tokenizer = AutoTokenizer.from_pretrained (MODEL_PATH)


#modelo.save_pretrained (r"C:\Users\Admin\Desktop\models\microsoftphi-4-Q4")
#tokenizer.save_pretrained (r"C:\Users\Admin\Desktop\models\microsoftphi-4-Q4")

<h1> Context Engineering com Constraint Decoding </h1>

In [ ]:
#Definir a estrutura do Constraint Decoding

class PARES (BaseModel):
    query: str
    chunk_id: List[int]

class DATASET (BaseModel):
    data: List[PARES]


model = outlines.from_transformers (modelo, tokenizer)

In [ ]:
x = 0
y = 15


for i in range (len(documentos) // 15 - 3):

    #Exemplo de Context Engineering
    prompt = f"""
    <s>system

    És um assistente de IA com a função de Senior Data Scientist que tens como único e principal objetivo 
    criar um dataset sintético de acordo com os dados fornecidos. Deves criar perguntas de acordo com o 
    contexto e selecionar quais são os chunks mais relevantes para responder a essas perguntas. 

    <REGRAS>

    1. És um modelo que retorna outputs apenas em formato JSON. Qualquer coisa retornada num formato diferente está errada!
    2. Deves ter em conta e ser fiel aos dados fornecidos.
    3. Cria perguntas concisas e diretas de acordo com o contexto fornecido.
    4. Deves ter em conta o 'chunk_id' de cada contexto e selecionar quais os 'chunk_id' mais relevantes para responder à pergunta criada. Tens que selecionar sempre mais que um 'chunk_id'!!
    5. Não podes repetir perguntas e retorna apenas 5 exemplos por prompt!
    6. As perguntas têm de estar em Português Europeu!

    </REGRAS>

    Tens como principal e única função responder de acordo com o contexto fornecido e criar perguntas 
    de acordo com o contexto e selecionar os chunk_id mais relevantes para responder à pergunta
    criada. Lembra-te que é fulcral que o output seja em formato JSON e em Português Europeu.
    </s>

    
    <s>user

    <CONTEXTO>

    {documentos[x:y]}

    </CONTEXTO
    </s>

    <s>assistant

    """

    result = model (prompt, output_type = DATASET, max_new_tokens = 512) #temp, top_k, top_p
    #display (result)


    with open ("dataset-Mistral.json", "a", encoding = "utf-8") as f:
        f.write (result + "\n") 
    
    x = y
    y = y + 15

    torch.cuda.empty_cache()